# Stage 2 — Nested Monthly Walk-Forward ETF Portfolio

This notebook constructs an independently estimated eight-ETF portfolio for every holding month from August 2016 onward. It uses fixed SOXX plus the seven Stage 1 monthly winners and never uses data later than the portfolio formation cutoff.

For each holding month, the model uses a five-year information set containing 24 rolling one-month validation folds. Every fold trains on the preceding 36 calendar months of weekly total returns. The best of three predefined risk profiles is selected on the stitched validation path, refitted on the latest 36 months, constrained to long-only 5%–25% positions (SOXX at least 10%), and rounded to 1% before the next month is held.

Expected run time on the current computer is approximately 3–5 minutes. Every run creates a new timestamped folder under `03_STAGE2/stage2_results`; prior runs are not overwritten.


## How to run

1. Open this notebook from the ETF portfolio project's `03_STAGE2` folder.
2. Run all cells from top to bottom.
3. If SciPy is missing, run `%pip install scipy matplotlib`, restart the kernel, and run all again.
4. The final cell prints the new run folder and creates CSV audit files, charts, a manifest, and `Stage2_ETF_Portfolio_Results.xlsx`.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "SciPy is required. Run `%pip install scipy` in a notebook cell, restart the kernel, and run again."
    ) from exc

warnings.filterwarnings(
    "ignore",
    message="Values in x were outside bounds during a minimize step, clipping to bounds",
    category=RuntimeWarning,
)


## Configuration

Editable model assumptions. These values define the windows, risk profiles, constraints, rounding rule, transaction cost, and drawdown threshold used in the run.


In [ ]:
LOOKBACK_MONTHS = 60
TRAIN_MONTHS = 36
INNER_VALIDATION_MONTHS = 24
WEEKLY_RULE = "W-FRI"
MIN_WEIGHT = 0.05
SOXX_MIN_WEIGHT = 0.10
MAX_WEIGHT = 0.25
ROUNDING_STEP = 0.01
NUMBER_OF_STARTS = 3
RANDOM_SEED = 20260805
TRANSACTION_COST_BPS = 10.0
SPY_DRAWDOWN_THRESHOLD = -0.15
MIN_WEEKLY_TRAIN_OBSERVATIONS = 140

SLEEVE_ORDER = [
    "Semiconductors",
    "Pharmaceuticals",
    "Consumer Staples",
    "Energy",
    "Utilities",
    "Capital Markets & Exchanges",
    "Aerospace & Defense",
    "Insurance",
]

STAGE1_COLUMNS = [
    "Pharmaceuticals",
    "Consumer Staples",
    "Energy",
    "Utilities",
    "Capital Markets & Exchanges",
    "Aerospace & Defense",
    "Insurance",
]

RISK_PROFILES = {
    "Balanced": {
        "return_reward": 1.00,
        "volatility_penalty": 0.20,
        "drawdown_penalty": 0.35,
        "spy_correlation_penalty": 0.05,
        "soxx_correlation_penalty": 0.05,
        "downside_excess_reward": 0.25,
        "equal_weight_regularization": 1.00,
    },
    "Defensive": {
        "return_reward": 0.75,
        "volatility_penalty": 0.45,
        "drawdown_penalty": 0.75,
        "spy_correlation_penalty": 0.08,
        "soxx_correlation_penalty": 0.08,
        "downside_excess_reward": 0.35,
        "equal_weight_regularization": 1.25,
    },
    "Growth": {
        "return_reward": 1.25,
        "volatility_penalty": 0.10,
        "drawdown_penalty": 0.20,
        "spy_correlation_penalty": 0.03,
        "soxx_correlation_penalty": 0.03,
        "downside_excess_reward": 0.15,
        "equal_weight_regularization": 0.75,
    },
}

VALIDATION_UTILITY = {
    "return_reward": 1.00,
    "volatility_penalty": 0.25,
    "drawdown_penalty": 0.50,
    "spy_correlation_penalty": 0.05,
    "soxx_correlation_penalty": 0.05,
    "downside_excess_reward": 0.25,
}


## Paths and source loading

Locate the ETF project, latest Stage 1 winner matrix, Yahoo adjusted-close files, and create a new immutable Stage 2 run folder.


In [ ]:
def discover_etf_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        Path(r"C:\Users\Kevin\Desktop\量化\finance homework\ETF portfolio"),
    ]
    for candidate in candidates:
        if (
            (candidate / "01_PRE_STAGE1" / "Final_Project_v7" / "yahoo_csv").is_dir()
            and (candidate / "02_STAGE1" / "stage1_results").is_dir()
            and (candidate / "03_STAGE2").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Cannot locate the ETF portfolio project. Run this notebook from 03_STAGE2 or set the path manually."
    )


PROJECT_ROOT = Path(os.environ["ETF_PROJECT_ROOT"]).resolve() if os.environ.get("ETF_PROJECT_ROOT") else discover_etf_project_root()
PRESTAGE_ROOT = PROJECT_ROOT / "01_PRE_STAGE1" / "Final_Project_v7"
STAGE1_PARENT = PROJECT_ROOT / "02_STAGE1" / "stage1_results"
STAGE2_ROOT = PROJECT_ROOT / "03_STAGE2"
YAHOO_DIR = PRESTAGE_ROOT / "yahoo_csv"
OUTPUT_PARENT = Path(os.environ["STAGE2_OUTPUT_PARENT"]).resolve() if os.environ.get("STAGE2_OUTPUT_PARENT") else STAGE2_ROOT / "stage2_results"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)


def latest_stage1_run() -> Path:
    runs = sorted(
        [p for p in STAGE1_PARENT.glob("run_*") if p.is_dir() and (p / "stage1_winner_matrix.csv").is_file()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not runs:
        raise FileNotFoundError("No completed Stage 1 run containing stage1_winner_matrix.csv was found.")
    return runs[0]


STAGE1_RUN = latest_stage1_run()
WINNER_MATRIX_PATH = STAGE1_RUN / "stage1_winner_matrix.csv"


def unique_run_dir() -> Path:
    stamp = datetime.now().strftime("run_%Y%m%d_%H%M%S")
    path = OUTPUT_PARENT / stamp
    suffix = 1
    while path.exists():
        path = OUTPUT_PARENT / f"{stamp}_{suffix:02d}"
        suffix += 1
    path.mkdir(parents=False, exist_ok=False)
    return path


RUN_DIR = unique_run_dir()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def load_adjusted_close(path: Path, ticker: str) -> pd.Series:
    data = pd.read_csv(path, encoding="utf-8-sig")
    normalized = {
        " ".join(str(column).replace("_", " ").strip().lower().split()): column
        for column in data.columns
    }
    required = {"date": "Date", "adj close": "Adj Close"}
    missing = [label for key, label in required.items() if key not in normalized]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; found {list(data.columns)}")
    data = data.rename(columns={normalized[key]: label for key, label in required.items()})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    data["Adj Close"] = pd.to_numeric(data["Adj Close"], errors="coerce")
    data = data.dropna(subset=["Date", "Adj Close"]).sort_values("Date")
    if data["Date"].duplicated().any() or (data["Adj Close"] <= 0).any():
        raise ValueError(f"{path.name}: duplicate dates or invalid adjusted prices detected")
    series = data.set_index("Date")["Adj Close"]
    series.name = ticker
    return series


winner_matrix = pd.read_csv(WINNER_MATRIX_PATH, dtype={"holding_month": str})
missing_columns = [column for column in ["holding_month", *STAGE1_COLUMNS] if column not in winner_matrix.columns]
if missing_columns:
    raise ValueError(f"Stage 1 winner matrix is missing columns: {missing_columns}")
if winner_matrix["holding_month"].duplicated().any():
    raise ValueError("Stage 1 winner matrix contains duplicate holding months")
winner_matrix["holding_period"] = pd.PeriodIndex(winner_matrix["holding_month"], freq="M")
winner_matrix = winner_matrix.sort_values("holding_period").reset_index(drop=True)

all_tickers = {"SOXX", "SPY"}
for column in STAGE1_COLUMNS:
    all_tickers.update(winner_matrix[column].dropna().astype(str).str.strip())

price_series = {}
source_rows = []
for ticker in sorted(all_tickers):
    path = YAHOO_DIR / f"{ticker}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing Yahoo CSV: {path}")
    series = load_adjusted_close(path, ticker)
    price_series[ticker] = series
    source_rows.append(
        {
            "ticker": ticker,
            "file": str(path),
            "sha256": file_sha256(path),
            "rows": len(series),
            "first_date": series.index.min().date().isoformat(),
            "last_date": series.index.max().date().isoformat(),
            "status": "OK",
        }
    )

daily_prices = pd.concat(price_series.values(), axis=1).sort_index()
weekly_prices = daily_prices.resample(WEEKLY_RULE).last()
weekly_returns = weekly_prices.pct_change(fill_method=None)
monthly_prices = daily_prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change(fill_method=None)
monthly_returns.index = monthly_returns.index.to_period("M")
source_quality = pd.DataFrame(source_rows)

print("ETF project root:", PROJECT_ROOT)
print("Stage 1 source run:", STAGE1_RUN)
print("New Stage 2 run:", RUN_DIR)
print("Holding months:", winner_matrix["holding_month"].min(), "to", winner_matrix["holding_month"].max())
print("Unique price files:", len(all_tickers))


## Optimizer functions

Define the constrained SLSQP optimizer, performance components, downside behavior, fixed risk-profile objectives, and exact 1% rounding routine.


In [ ]:
def stable_rng(*parts) -> np.random.Generator:
    text = "|".join(map(str, parts))
    offset = int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:8], 16)
    return np.random.default_rng((RANDOM_SEED + offset) % (2**32 - 1))


def bounds_for_assets(assets: list[str]) -> list[tuple[float, float]]:
    return [
        (SOXX_MIN_WEIGHT if ticker == "SOXX" else MIN_WEIGHT, MAX_WEIGHT)
        for ticker in assets
    ]


def validate_weights(weights: np.ndarray, assets: list[str], tolerance: float = 1e-7) -> bool:
    weights = np.asarray(weights, dtype=float)
    bounds = bounds_for_assets(assets)
    return (
        len(weights) == len(assets)
        and np.isfinite(weights).all()
        and abs(weights.sum() - 1.0) <= tolerance
        and all(lower - tolerance <= weight <= upper + tolerance for weight, (lower, upper) in zip(weights, bounds))
    )


def project_to_constraints(target: np.ndarray, assets: list[str]) -> np.ndarray:
    target = np.asarray(target, dtype=float)
    equal = np.full(len(assets), 1.0 / len(assets))
    result = minimize(
        lambda weights: float(np.sum((weights - target) ** 2)),
        x0=equal,
        method="SLSQP",
        bounds=bounds_for_assets(assets),
        constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}],
        options={"ftol": 1e-12, "maxiter": 500, "disp": False},
    )
    if not result.success or not validate_weights(result.x, assets):
        raise RuntimeError("Could not project weights into the feasible region")
    return result.x


def round_weights_largest_remainder(weights: np.ndarray, assets: list[str]) -> np.ndarray:
    total_units = int(round(1.0 / ROUNDING_STEP))
    raw_units = np.asarray(weights, dtype=float) / ROUNDING_STEP
    bounds = bounds_for_assets(assets)
    min_units = np.asarray([math.ceil(lower / ROUNDING_STEP - 1e-12) for lower, _ in bounds], dtype=int)
    max_units = np.asarray([math.floor(upper / ROUNDING_STEP + 1e-12) for _, upper in bounds], dtype=int)
    units = np.clip(np.floor(raw_units).astype(int), min_units, max_units)
    while units.sum() < total_units:
        eligible = np.where(units < max_units)[0]
        if len(eligible) == 0:
            raise RuntimeError("Rounding constraints cannot reach 100%")
        residuals = raw_units[eligible] - units[eligible]
        choice = eligible[np.argmax(residuals)]
        units[choice] += 1
    while units.sum() > total_units:
        eligible = np.where(units > min_units)[0]
        if len(eligible) == 0:
            raise RuntimeError("Rounding constraints cannot reach 100%")
        excess = units[eligible] - raw_units[eligible]
        choice = eligible[np.argmax(excess)]
        units[choice] -= 1
    rounded = units.astype(float) * ROUNDING_STEP
    if not validate_weights(rounded, assets):
        raise RuntimeError("Rounded weights violate constraints")
    return rounded


def return_metrics(
    portfolio_returns: pd.Series,
    spy_returns: pd.Series,
    soxx_returns: pd.Series,
    periods_per_year: int,
) -> dict[str, float]:
    aligned = pd.concat(
        [
            portfolio_returns.rename("Portfolio"),
            spy_returns.rename("SPY"),
            soxx_returns.rename("SOXX"),
        ],
        axis=1,
        join="inner",
    ).dropna()
    if len(aligned) < 2:
        return {key: np.nan for key in [
            "CAGR", "Volatility", "Maximum drawdown", "Correlation SPY", "Correlation SOXX",
            "Downside excess", "Sharpe", "Sortino", "Beta SPY", "Downside beta SPY", "Downside capture",
        ]}
    nav = (1.0 + aligned["Portfolio"]).cumprod()
    cagr_value = float(nav.iloc[-1] ** (periods_per_year / len(nav)) - 1.0)
    volatility = float(aligned["Portfolio"].std(ddof=1) * math.sqrt(periods_per_year))
    max_drawdown = float((nav / nav.cummax() - 1.0).min())
    corr_spy = float(aligned["Portfolio"].corr(aligned["SPY"]))
    corr_soxx = float(aligned["Portfolio"].corr(aligned["SOXX"]))
    mean_return = float(aligned["Portfolio"].mean() * periods_per_year)
    sharpe = mean_return / volatility if volatility > 0 else np.nan
    downside_values = aligned.loc[aligned["Portfolio"] < 0, "Portfolio"]
    downside_deviation = float(np.sqrt(np.mean(np.square(downside_values))) * math.sqrt(periods_per_year)) if len(downside_values) else np.nan
    sortino = mean_return / downside_deviation if downside_deviation and downside_deviation > 0 else np.nan
    spy_variance = float(aligned["SPY"].var(ddof=1))
    beta = float(aligned["Portfolio"].cov(aligned["SPY"]) / spy_variance) if spy_variance > 0 else np.nan
    negative_spy = aligned["SPY"] < 0
    down = aligned.loc[negative_spy]
    if len(down) >= 2 and float(down["SPY"].var(ddof=1)) > 0:
        downside_beta = float(down["Portfolio"].cov(down["SPY"]) / down["SPY"].var(ddof=1))
    else:
        downside_beta = np.nan
    downside_capture = float(down["Portfolio"].mean() / down["SPY"].mean()) if len(down) and down["SPY"].mean() != 0 else np.nan
    downside_excess = float((down["Portfolio"] - down["SPY"]).mean() * periods_per_year) if len(down) else 0.0
    return {
        "CAGR": cagr_value,
        "Volatility": volatility,
        "Maximum drawdown": max_drawdown,
        "Correlation SPY": corr_spy,
        "Correlation SOXX": corr_soxx,
        "Downside excess": downside_excess,
        "Sharpe": float(sharpe),
        "Sortino": float(sortino),
        "Beta SPY": beta,
        "Downside beta SPY": downside_beta,
        "Downside capture": downside_capture,
    }


def utility_from_metrics(metrics: dict[str, float], coefficients: dict[str, float], concentration: float = 0.0) -> float:
    required = ["CAGR", "Volatility", "Maximum drawdown", "Correlation SPY", "Correlation SOXX", "Downside excess"]
    if any(pd.isna(metrics[key]) for key in required):
        return -1e6
    return float(
        coefficients["return_reward"] * metrics["CAGR"]
        - coefficients["volatility_penalty"] * metrics["Volatility"]
        - coefficients["drawdown_penalty"] * abs(metrics["Maximum drawdown"])
        - coefficients["spy_correlation_penalty"] * max(metrics["Correlation SPY"], 0.0)
        - coefficients["soxx_correlation_penalty"] * max(metrics["Correlation SOXX"], 0.0)
        + coefficients["downside_excess_reward"] * metrics["Downside excess"]
        - coefficients.get("equal_weight_regularization", 0.0) * concentration
    )


def optimize_training_window(
    train_returns: pd.DataFrame,
    assets: list[str],
    profile_name: str,
    key_parts: tuple,
) -> tuple[np.ndarray, float, int, int]:
    required = [*assets, "SPY"]
    train = train_returns[required].dropna()
    if len(train) < MIN_WEEKLY_TRAIN_OBSERVATIONS:
        raise ValueError(f"Only {len(train)} weekly observations; requires {MIN_WEEKLY_TRAIN_OBSERVATIONS}")
    coefficients = RISK_PROFILES[profile_name]
    asset_matrix = train[assets].to_numpy(dtype=float)
    spy_array = train["SPY"].to_numpy(dtype=float)
    soxx_array = train["SOXX"].to_numpy(dtype=float)
    equal = np.full(len(assets), 1.0 / len(assets))
    inv_vol = 1.0 / train[assets].std(ddof=1).replace(0.0, np.nan).to_numpy(dtype=float)
    inv_vol = np.nan_to_num(inv_vol, nan=1.0, posinf=1.0, neginf=1.0)
    inv_vol /= inv_vol.sum()
    inv_vol = project_to_constraints(inv_vol, assets)
    rng = stable_rng(*key_parts, profile_name)
    starts = [equal, inv_vol]
    while len(starts) < NUMBER_OF_STARTS:
        random_target = 0.70 * equal + 0.30 * rng.dirichlet(np.ones(len(assets)))
        starts.append(project_to_constraints(random_target, assets))

    def objective(weights):
        if not validate_weights(weights, assets, tolerance=1e-5):
            return 1e6
        portfolio = asset_matrix @ np.asarray(weights, dtype=float)
        gross = 1.0 + portfolio
        if np.any(gross <= 0.0):
            return 1e6
        nav = np.cumprod(gross)
        cagr_value = float(nav[-1] ** (52.0 / len(nav)) - 1.0)
        volatility = float(np.std(portfolio, ddof=1) * math.sqrt(52.0))
        maximum_drawdown = float(np.min(nav / np.maximum.accumulate(nav) - 1.0))
        corr_spy = float(np.corrcoef(portfolio, spy_array)[0, 1])
        corr_soxx = float(np.corrcoef(portfolio, soxx_array)[0, 1])
        negative_spy = spy_array < 0.0
        downside_excess = float(np.mean(portfolio[negative_spy] - spy_array[negative_spy]) * 52.0) if np.any(negative_spy) else 0.0
        metrics = {
            "CAGR": cagr_value,
            "Volatility": volatility,
            "Maximum drawdown": maximum_drawdown,
            "Correlation SPY": corr_spy,
            "Correlation SOXX": corr_soxx,
            "Downside excess": downside_excess,
        }
        concentration = float(np.sum((np.asarray(weights) - equal) ** 2))
        return -utility_from_metrics(metrics, coefficients, concentration)

    candidates = []
    for start_number, start in enumerate(starts, start=1):
        result = minimize(
            objective,
            x0=start,
            method="SLSQP",
            bounds=bounds_for_assets(assets),
            constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}],
            options={"maxiter": 250, "ftol": 1e-9, "disp": False},
        )
        if result.success and validate_weights(result.x, assets):
            candidates.append((float(result.fun), result.x.copy(), start_number))
    if not candidates:
        raise RuntimeError(f"No optimizer start converged for {profile_name} {key_parts}")
    candidates.sort(key=lambda item: (item[0], item[2]))
    best_objective, best_weights, best_start = candidates[0]
    rounded = round_weights_largest_remainder(best_weights, assets)
    return rounded, -best_objective, best_start, len(candidates)


## Nested monthly walk-forward calculation

For every outer holding month, run 24 inner validation folds, choose the best predefined profile, and refit it using only information available at the formation cutoff.


In [ ]:
fold_cache: dict[tuple, dict] = {}
final_weight_rows = []
inner_summary_rows = []
inner_fold_rows = []
optimizer_rows = []


def assets_for_winner_row(row) -> tuple[list[str], dict[str, str]]:
    sleeve_tickers = {"Semiconductors": "SOXX"}
    for sleeve in STAGE1_COLUMNS:
        sleeve_tickers[sleeve] = str(getattr(row, sleeve.replace(" ", "_").replace("&", ""), ""))
    # itertuples sanitizes names unpredictably for ampersands, so reconstruct by Series elsewhere.
    assets = [sleeve_tickers[sleeve] for sleeve in SLEEVE_ORDER]
    return assets, sleeve_tickers


for outer_number, row in winner_matrix.iterrows():
    holding_period = row["holding_period"]
    holding_month = str(holding_period)
    formation_cutoff = holding_period.start_time - pd.Timedelta(days=1)
    sleeve_tickers = {"Semiconductors": "SOXX", **{sleeve: str(row[sleeve]).strip() for sleeve in STAGE1_COLUMNS}}
    assets = [sleeve_tickers[sleeve] for sleeve in SLEEVE_ORDER]
    if len(set(assets)) != len(assets):
        raise ValueError(f"{holding_month}: duplicate ETF appears in more than one sleeve: {assets}")

    profile_results = []
    validation_periods = pd.period_range(holding_period - INNER_VALIDATION_MONTHS, holding_period - 1, freq="M")
    if len(validation_periods) != INNER_VALIDATION_MONTHS:
        raise AssertionError("Unexpected number of validation months")

    for profile_name in RISK_PROFILES:
        profile_portfolio_returns = []
        profile_spy_returns = []
        profile_soxx_returns = []
        for validation_period in validation_periods:
            train_start = validation_period.start_time - pd.DateOffset(months=TRAIN_MONTHS)
            train_end = validation_period.start_time - pd.Timedelta(days=1)
            cache_key = (tuple(assets), str(validation_period), profile_name)
            if cache_key not in fold_cache:
                train = weekly_returns.loc[(weekly_returns.index >= train_start) & (weekly_returns.index <= train_end)]
                weights, training_utility, best_start, converged = optimize_training_window(
                    train,
                    assets,
                    profile_name,
                    (tuple(assets), str(validation_period), train_start.date(), train_end.date()),
                )
                validation_vector = monthly_returns.loc[validation_period, assets]
                if validation_vector.isna().any():
                    raise ValueError(f"{validation_period}: missing monthly return for {assets}")
                portfolio_return = float(np.dot(weights, validation_vector.to_numpy(dtype=float)))
                fold_cache[cache_key] = {
                    "weights": weights,
                    "training_utility": training_utility,
                    "best_start": best_start,
                    "converged_starts": converged,
                    "training_start": train.index.min(),
                    "training_end": train.index.max(),
                    "weekly_observations": len(train[[*assets, "SPY"]].dropna()),
                    "portfolio_return": portfolio_return,
                    "spy_return": float(monthly_returns.loc[validation_period, "SPY"]),
                    "soxx_return": float(monthly_returns.loc[validation_period, "SOXX"]),
                }
            fold = fold_cache[cache_key]
            profile_portfolio_returns.append(fold["portfolio_return"])
            profile_spy_returns.append(fold["spy_return"])
            profile_soxx_returns.append(fold["soxx_return"])
            fold_row = {
                "outer_holding_month": holding_month,
                "formation_cutoff": formation_cutoff.date().isoformat(),
                "profile": profile_name,
                "validation_month": str(validation_period),
                "training_start": pd.Timestamp(fold["training_start"]).date().isoformat(),
                "training_end": pd.Timestamp(fold["training_end"]).date().isoformat(),
                "weekly_observations": fold["weekly_observations"],
                "training_utility": fold["training_utility"],
                "validation_portfolio_return": fold["portfolio_return"],
                "validation_spy_return": fold["spy_return"],
                "validation_soxx_return": fold["soxx_return"],
                "winning_start": fold["best_start"],
                "converged_starts": fold["converged_starts"],
            }
            for ticker, weight in zip(assets, fold["weights"]):
                fold_row[f"weight_{ticker}"] = weight
            inner_fold_rows.append(fold_row)

        validation_index = validation_periods.astype(str)
        portfolio_series = pd.Series(profile_portfolio_returns, index=validation_index, dtype=float)
        spy_series = pd.Series(profile_spy_returns, index=validation_index, dtype=float)
        soxx_series = pd.Series(profile_soxx_returns, index=validation_index, dtype=float)
        validation_metrics = return_metrics(portfolio_series, spy_series, soxx_series, periods_per_year=12)
        validation_utility = utility_from_metrics(validation_metrics, VALIDATION_UTILITY)
        summary_row = {
            "holding_month": holding_month,
            "formation_cutoff": formation_cutoff.date().isoformat(),
            "profile": profile_name,
            "validation_start": str(validation_periods.min()),
            "validation_end": str(validation_periods.max()),
            "validation_months": len(validation_periods),
            "validation_utility": validation_utility,
            **{f"validation_{key}": value for key, value in validation_metrics.items()},
        }
        inner_summary_rows.append(summary_row)
        profile_results.append(summary_row)

    profile_results.sort(
        key=lambda result: (
            -result["validation_utility"],
            -result["validation_Maximum drawdown"],
            result["validation_Volatility"],
            result["profile"],
        )
    )
    selected = profile_results[0]
    selected_profile = selected["profile"]

    final_train_start = holding_period.start_time - pd.DateOffset(months=TRAIN_MONTHS)
    final_train_end = formation_cutoff
    final_train = weekly_returns.loc[(weekly_returns.index >= final_train_start) & (weekly_returns.index <= final_train_end)]
    final_weights, final_training_utility, best_start, converged = optimize_training_window(
        final_train,
        assets,
        selected_profile,
        (tuple(assets), holding_month, "final_refit", final_train_start.date(), final_train_end.date()),
    )
    weight_row = {
        "holding_month": holding_month,
        "formation_cutoff": formation_cutoff.date().isoformat(),
        "selected_profile": selected_profile,
        "selected_profile_validation_utility": selected["validation_utility"],
        "final_training_start": final_train.index.min().date().isoformat(),
        "final_training_end": final_train.index.max().date().isoformat(),
        "final_training_weekly_observations": len(final_train[[*assets, "SPY"]].dropna()),
        "weight_sum": float(final_weights.sum()),
    }
    for sleeve, ticker, weight in zip(SLEEVE_ORDER, assets, final_weights):
        safe = sleeve.lower().replace(" ", "_").replace("&", "and")
        weight_row[f"{safe}_ticker"] = ticker
        weight_row[f"{safe}_weight"] = float(weight)
    final_weight_rows.append(weight_row)
    optimizer_rows.append(
        {
            "holding_month": holding_month,
            "selected_profile": selected_profile,
            "final_training_utility": final_training_utility,
            "winning_start": best_start,
            "converged_starts": converged,
            "minimum_weight": float(final_weights.min()),
            "maximum_weight": float(final_weights.max()),
            "weight_sum": float(final_weights.sum()),
        }
    )
    if (outer_number + 1) % 12 == 0 or outer_number == 0 or outer_number + 1 == len(winner_matrix):
        print(f"Completed monthly weight {outer_number + 1}/{len(winner_matrix)}: {holding_month} ({selected_profile})")

monthly_weights = pd.DataFrame(final_weight_rows)
inner_validation_summary = pd.DataFrame(inner_summary_rows)
inner_fold_results = pd.DataFrame(inner_fold_rows)
optimizer_diagnostics = pd.DataFrame(optimizer_rows)


## Daily and monthly out-of-sample portfolio simulation

Stitch each one-month holding period into a continuous gross and transaction-cost-adjusted net portfolio path while accounting for turnover when tickers or weights change.


In [ ]:
def target_weight_map(weight_row: pd.Series) -> dict[str, float]:
    result = {}
    for sleeve in SLEEVE_ORDER:
        safe = sleeve.lower().replace(" ", "_").replace("&", "and")
        result[str(weight_row[f"{safe}_ticker"])] = float(weight_row[f"{safe}_weight"])
    return result


gross_nav = 100.0
net_nav = 100.0
initial_period = pd.Period(monthly_weights.iloc[0]["holding_month"], freq="M")
initial_cutoff = initial_period.start_time - pd.Timedelta(days=1)
benchmark_base_prices = daily_prices.loc[:initial_cutoff, ["SPY", "SOXX"]].dropna().iloc[-1]
initial_date = daily_prices.loc[:initial_cutoff, ["SPY", "SOXX"]].dropna().index[-1]
daily_rows = [
    {
        "date": initial_date,
        "portfolio_gross_nav": gross_nav,
        "portfolio_net_nav": net_nav,
        "spy_nav": 100.0,
        "soxx_nav": 100.0,
    }
]
monthly_performance_rows = []
previous_end_weights: dict[str, float] | None = None

for _, weight_row in monthly_weights.iterrows():
    holding_period = pd.Period(weight_row["holding_month"], freq="M")
    target = target_weight_map(weight_row)
    assets = list(target)
    required_month_columns = list(dict.fromkeys([*assets, "SPY", "SOXX"]))
    cutoff = holding_period.start_time - pd.Timedelta(days=1)
    available_base = daily_prices.loc[:cutoff, required_month_columns].dropna()
    if available_base.empty:
        raise ValueError(f"{holding_period}: no aligned base prices")
    base_date = available_base.index[-1]
    base_prices = available_base.iloc[-1]
    actual_end = min(holding_period.end_time.normalize(), daily_prices.index.max())
    month_prices = daily_prices.loc[
        (daily_prices.index > base_date) & (daily_prices.index <= actual_end),
        required_month_columns,
    ].dropna()
    if month_prices.empty:
        raise ValueError(f"{holding_period}: no holding-period prices")

    if previous_end_weights is None:
        turnover = 1.0
    else:
        union = set(previous_end_weights) | set(target)
        turnover = 0.5 * sum(abs(target.get(ticker, 0.0) - previous_end_weights.get(ticker, 0.0)) for ticker in union)
    cost_rate = turnover * TRANSACTION_COST_BPS / 10_000.0
    gross_start = gross_nav
    net_start_after_cost = net_nav * (1.0 - cost_rate)

    relative = month_prices[assets].div(base_prices[assets], axis=1)
    target_vector = np.asarray([target[ticker] for ticker in assets], dtype=float)
    month_multiplier = relative.to_numpy(dtype=float) @ target_vector
    gross_path = gross_start * month_multiplier
    net_path = net_start_after_cost * month_multiplier
    spy_path = month_prices["SPY"] / benchmark_base_prices["SPY"] * 100.0
    soxx_path = month_prices["SOXX"] / benchmark_base_prices["SOXX"] * 100.0
    for date, gross_value, net_value, spy_value, soxx_value in zip(
        month_prices.index, gross_path, net_path, spy_path, soxx_path
    ):
        daily_rows.append(
            {
                "date": date,
                "portfolio_gross_nav": float(gross_value),
                "portfolio_net_nav": float(net_value),
                "spy_nav": float(spy_value),
                "soxx_nav": float(soxx_value),
            }
        )

    gross_nav = float(gross_path[-1])
    net_nav = float(net_path[-1])
    asset_period_returns = month_prices[assets].iloc[-1].div(base_prices[assets]) - 1.0
    gross_return = float(np.dot(target_vector, asset_period_returns.to_numpy(dtype=float)))
    net_return = float((1.0 - cost_rate) * (1.0 + gross_return) - 1.0)
    spy_return = float(month_prices["SPY"].iloc[-1] / base_prices["SPY"] - 1.0)
    soxx_return = float(month_prices["SOXX"].iloc[-1] / base_prices["SOXX"] - 1.0)
    ending_values = target_vector * (1.0 + asset_period_returns.to_numpy(dtype=float))
    ending_values /= ending_values.sum()
    previous_end_weights = {ticker: float(value) for ticker, value in zip(assets, ending_values)}
    monthly_performance_rows.append(
        {
            "holding_month": str(holding_period),
            "period_start_base_date": base_date.date().isoformat(),
            "period_end_date": month_prices.index[-1].date().isoformat(),
            "is_partial_month": month_prices.index[-1].to_period("M") == daily_prices.index.max().to_period("M")
            and daily_prices.index.max().normalize() < holding_period.end_time.normalize(),
            "selected_profile": weight_row["selected_profile"],
            "portfolio_gross_return": gross_return,
            "portfolio_net_return": net_return,
            "spy_return": spy_return,
            "soxx_return": soxx_return,
            "turnover": turnover,
            "transaction_cost_rate": cost_rate,
            "portfolio_gross_nav": gross_nav,
            "portfolio_net_nav": net_nav,
            "spy_nav": float(spy_path.iloc[-1]),
            "soxx_nav": float(soxx_path.iloc[-1]),
        }
    )

daily_nav = pd.DataFrame(daily_rows).drop_duplicates("date", keep="last").sort_values("date").reset_index(drop=True)
monthly_performance = pd.DataFrame(monthly_performance_rows)
for nav_column in ["portfolio_gross_nav", "portfolio_net_nav", "spy_nav", "soxx_nav"]:
    daily_nav[nav_column.replace("_nav", "_drawdown")] = daily_nav[nav_column] / daily_nav[nav_column].cummax() - 1.0


## Performance, correlation, and SPY drawdown-event analysis

Calculate cumulative return, CAGR, volatility, Sharpe, Sortino, maximum drawdown, monthly correlations, and portfolio behavior during SPY drawdown events greater than 15%.


In [ ]:
def nav_performance_metrics(nav: pd.Series, dates: pd.Series) -> dict[str, float]:
    nav = pd.Series(nav.to_numpy(dtype=float), index=pd.to_datetime(dates)).dropna()
    daily_return = nav.pct_change(fill_method=None).dropna()
    monthly_nav = nav.resample("ME").last()
    monthly_return = monthly_nav.pct_change(fill_method=None).dropna()
    years = (nav.index[-1] - nav.index[0]).days / 365.2425
    cagr_value = float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / years) - 1.0) if years > 0 else np.nan
    daily_vol = float(daily_return.std(ddof=1) * math.sqrt(252))
    annual_mean = float(daily_return.mean() * 252)
    downside = daily_return[daily_return < 0]
    downside_dev = float(np.sqrt(np.mean(np.square(downside))) * math.sqrt(252)) if len(downside) else np.nan
    return {
        "Cumulative return": float(nav.iloc[-1] / nav.iloc[0] - 1.0),
        "CAGR": cagr_value,
        "Annualized daily volatility": daily_vol,
        "Sharpe ratio (rf=0)": annual_mean / daily_vol if daily_vol > 0 else np.nan,
        "Sortino ratio (rf=0)": annual_mean / downside_dev if downside_dev and downside_dev > 0 else np.nan,
        "Maximum daily drawdown": float((nav / nav.cummax() - 1.0).min()),
        "Best month": float(monthly_return.max()),
        "Worst month": float(monthly_return.min()),
        "Positive month ratio": float((monthly_return > 0).mean()),
    }


series_map = {
    "Portfolio Gross": "portfolio_gross_nav",
    "Portfolio Net": "portfolio_net_nav",
    "SPY": "spy_nav",
    "SOXX": "soxx_nav",
}
summary_dict = {
    name: nav_performance_metrics(daily_nav[column], daily_nav["date"])
    for name, column in series_map.items()
}

monthly_return_table = monthly_performance.set_index("holding_month")[[
    "portfolio_gross_return", "portfolio_net_return", "spy_return", "soxx_return"
]].rename(columns={
    "portfolio_gross_return": "Portfolio Gross",
    "portfolio_net_return": "Portfolio Net",
    "spy_return": "SPY",
    "soxx_return": "SOXX",
})
correlation_matrix = monthly_return_table.corr()

portfolio_month_metrics = return_metrics(
    monthly_return_table["Portfolio Net"],
    monthly_return_table["SPY"],
    monthly_return_table["SOXX"],
    periods_per_year=12,
)
summary_dict["Portfolio Net"].update({
    "Monthly correlation with SPY": portfolio_month_metrics["Correlation SPY"],
    "Monthly correlation with SOXX": portfolio_month_metrics["Correlation SOXX"],
    "Monthly beta vs SPY": portfolio_month_metrics["Beta SPY"],
    "Monthly downside beta vs SPY": portfolio_month_metrics["Downside beta SPY"],
    "Monthly downside capture ratio": portfolio_month_metrics["Downside capture"],
    "Average monthly turnover": float(monthly_performance["turnover"].mean()),
    "Annualized turnover": float(monthly_performance["turnover"].mean() * 12),
    "Gross-to-net cumulative cost drag": float(
        summary_dict["Portfolio Gross"]["Cumulative return"] - summary_dict["Portfolio Net"]["Cumulative return"]
    ),
})

performance_summary = pd.DataFrame(summary_dict)
performance_summary.index.name = "metric"


def identify_spy_drawdown_events(nav_frame: pd.DataFrame, threshold: float) -> pd.DataFrame:
    frame = nav_frame.set_index(pd.to_datetime(nav_frame["date"]))
    spy = frame["spy_nav"]
    spy_drawdown = spy / spy.cummax() - 1.0
    events = []
    in_event = False
    start_date = None
    peak_value = None
    for date, dd in spy_drawdown.items():
        if not in_event and dd <= threshold:
            peak_slice = spy.loc[:date]
            peak_value = peak_slice.cummax().iloc[-1]
            start_date = peak_slice.loc[peak_slice == peak_value].index[-1]
            in_event = True
        if in_event:
            recovered = spy.loc[date] >= peak_value - 1e-12
            if recovered and date > start_date:
                end_date = date
                segment = frame.loc[start_date:end_date]
                trough_date = segment["spy_nav"].idxmin()
                events.append((start_date, trough_date, end_date, True, segment))
                in_event = False
                start_date = None
                peak_value = None
    if in_event and start_date is not None:
        end_date = frame.index[-1]
        segment = frame.loc[start_date:end_date]
        trough_date = segment["spy_nav"].idxmin()
        events.append((start_date, trough_date, end_date, False, segment))

    rows = []
    for event_number, (start, trough, end, recovered, segment) in enumerate(events, start=1):
        start_values = segment.iloc[0]
        trough_values = segment.loc[trough]
        end_values = segment.iloc[-1]
        rows.append(
            {
                "event": event_number,
                "start_date": start.date().isoformat(),
                "trough_date": trough.date().isoformat(),
                "recovery_or_end_date": end.date().isoformat(),
                "recovered": recovered,
                "spy_max_drawdown": float((segment["spy_nav"] / segment["spy_nav"].cummax() - 1.0).min()),
                "portfolio_net_max_drawdown": float((segment["portfolio_net_nav"] / segment["portfolio_net_nav"].cummax() - 1.0).min()),
                "soxx_max_drawdown": float((segment["soxx_nav"] / segment["soxx_nav"].cummax() - 1.0).min()),
                "spy_peak_to_trough_return": float(trough_values["spy_nav"] / start_values["spy_nav"] - 1.0),
                "portfolio_net_return_at_spy_trough": float(trough_values["portfolio_net_nav"] / start_values["portfolio_net_nav"] - 1.0),
                "soxx_return_at_spy_trough": float(trough_values["soxx_nav"] / start_values["soxx_nav"] - 1.0),
                "portfolio_net_full_event_return": float(end_values["portfolio_net_nav"] / start_values["portfolio_net_nav"] - 1.0),
            }
        )
    return pd.DataFrame(rows)


spy_drawdown_events = identify_spy_drawdown_events(daily_nav, SPY_DRAWDOWN_THRESHOLD)


## Audit checks, exports, charts, and immutable manifest

Write every intermediate and final result, verify no future data was used, build the Excel report, hash source files, and package the run without overwriting prior runs.


In [ ]:
weight_columns = [column for column in monthly_weights.columns if column.endswith("_weight")]
audit_checks = [
    {
        "check": "Expected monthly weight rows",
        "actual": len(monthly_weights),
        "expected": len(winner_matrix),
        "tolerance": 0,
        "status": "OK" if len(monthly_weights) == len(winner_matrix) else "FAIL",
        "notes": "One independently trained target-weight row per Stage 1 holding month",
    },
    {
        "check": "Maximum absolute weight-sum error",
        "actual": float((monthly_weights[weight_columns].sum(axis=1) - 1.0).abs().max()),
        "expected": 0.0,
        "tolerance": 1e-9,
        "status": "OK" if (monthly_weights[weight_columns].sum(axis=1) - 1.0).abs().max() <= 1e-9 else "FAIL",
        "notes": "Rounded starting weights must sum to 100%",
    },
    {
        "check": "Minimum number of inner validation months",
        "actual": int(inner_validation_summary["validation_months"].min()),
        "expected": INNER_VALIDATION_MONTHS,
        "tolerance": 0,
        "status": "OK" if int(inner_validation_summary["validation_months"].min()) == INNER_VALIDATION_MONTHS else "FAIL",
        "notes": "Every profile-month decision uses 24 one-month validation folds",
    },
    {
        "check": "Future-data violations in final refits",
        "actual": int((pd.to_datetime(monthly_weights["final_training_end"]) > pd.to_datetime(monthly_weights["formation_cutoff"])).sum()),
        "expected": 0,
        "tolerance": 0,
        "status": "OK" if (pd.to_datetime(monthly_weights["final_training_end"]) <= pd.to_datetime(monthly_weights["formation_cutoff"])).all() else "FAIL",
        "notes": "Final weight estimation ends no later than the preceding month-end",
    },
    {
        "check": "Daily/monthly net NAV ending tie-out",
        "actual": float(abs(daily_nav.iloc[-1]["portfolio_net_nav"] - monthly_performance.iloc[-1]["portfolio_net_nav"])),
        "expected": 0.0,
        "tolerance": 1e-8,
        "status": "OK" if abs(daily_nav.iloc[-1]["portfolio_net_nav"] - monthly_performance.iloc[-1]["portfolio_net_nav"]) <= 1e-8 else "FAIL",
        "notes": "Independent daily and monthly paths end at the same NAV",
    },
]
audit_checks = pd.DataFrame(audit_checks)
if (audit_checks["status"] != "OK").any():
    raise AssertionError("Stage 2 audit failed:\n" + audit_checks.to_string(index=False))

settings_rows = []
for name, value in {
    "Generated at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "Stage 1 run": str(STAGE1_RUN),
    "Outer holding start": winner_matrix["holding_month"].min(),
    "Outer holding end": winner_matrix["holding_month"].max(),
    "Five-year information set (months)": LOOKBACK_MONTHS,
    "Inner training window (months)": TRAIN_MONTHS,
    "Inner validation months": INNER_VALIDATION_MONTHS,
    "Training return frequency": WEEKLY_RULE,
    "Minimum weight": MIN_WEIGHT,
    "SOXX minimum weight": SOXX_MIN_WEIGHT,
    "Maximum weight": MAX_WEIGHT,
    "Rounding step": ROUNDING_STEP,
    "Optimizer starts": NUMBER_OF_STARTS,
    "Random seed": RANDOM_SEED,
    "Transaction cost (bps per unit turnover)": TRANSACTION_COST_BPS,
    "SPY drawdown event threshold": SPY_DRAWDOWN_THRESHOLD,
}.items():
    settings_rows.append({"setting": name, "value": value})
for profile_name, coefficients in RISK_PROFILES.items():
    for coefficient, value in coefficients.items():
        settings_rows.append({"setting": f"Profile {profile_name}: {coefficient}", "value": value})
for coefficient, value in VALIDATION_UTILITY.items():
    settings_rows.append({"setting": f"Validation utility: {coefficient}", "value": value})
optimizer_settings = pd.DataFrame(settings_rows)

monthly_weights.to_csv(RUN_DIR / "monthly_starting_weights.csv", index=False)
monthly_performance.to_csv(RUN_DIR / "monthly_portfolio_performance.csv", index=False)
daily_nav.to_csv(RUN_DIR / "daily_portfolio_nav.csv", index=False)
performance_summary.to_csv(RUN_DIR / "performance_summary.csv")
correlation_matrix.to_csv(RUN_DIR / "monthly_return_correlation_matrix.csv")
spy_drawdown_events.to_csv(RUN_DIR / "spy_drawdown_events_over_15pct.csv", index=False)
inner_validation_summary.to_csv(RUN_DIR / "inner_validation_summary.csv", index=False)
inner_fold_results.to_csv(RUN_DIR / "inner_validation_fold_results.csv", index=False)
optimizer_diagnostics.to_csv(RUN_DIR / "final_optimizer_diagnostics.csv", index=False)
optimizer_settings.to_csv(RUN_DIR / "optimizer_settings.csv", index=False)
audit_checks.to_csv(RUN_DIR / "audit_checks.csv", index=False)
source_quality.to_csv(RUN_DIR / "source_quality.csv", index=False)

try:
    import matplotlib.pyplot as plt

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(12, 6))
    for label, column, color in [
        ("Portfolio Net", "portfolio_net_nav", "#0F766E"),
        ("Portfolio Gross", "portfolio_gross_nav", "#2563EB"),
        ("SPY", "spy_nav", "#475569"),
        ("SOXX", "soxx_nav", "#F59E0B"),
    ]:
        ax.plot(pd.to_datetime(daily_nav["date"]), daily_nav[column], label=label, linewidth=1.5, color=color)
    ax.set_title("Stage 2 Out-of-Sample Growth of $100")
    ax.set_ylabel("NAV")
    ax.legend(ncol=4, frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "portfolio_vs_benchmarks.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pd.to_datetime(daily_nav["date"]), daily_nav["portfolio_net_drawdown"], label="Portfolio Net", color="#0F766E")
    ax.plot(pd.to_datetime(daily_nav["date"]), daily_nav["spy_drawdown"], label="SPY", color="#475569")
    ax.plot(pd.to_datetime(daily_nav["date"]), daily_nav["soxx_drawdown"], label="SOXX", color="#F59E0B")
    ax.axhline(SPY_DRAWDOWN_THRESHOLD, color="#DC2626", linestyle="--", linewidth=1.0, label="-15% threshold")
    ax.set_title("Daily Drawdowns")
    ax.set_ylabel("Drawdown")
    ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
    ax.legend(ncol=4, frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "daily_drawdowns.png", dpi=160)
    plt.close(fig)
except Exception as chart_error:
    print("Chart generation warning:", chart_error)

payload = {
    "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "stage1_run": str(STAGE1_RUN),
    "tables": {
        "monthly_weights": monthly_weights.replace({np.nan: None}).to_dict("records"),
        "monthly_performance": monthly_performance.replace({np.nan: None}).to_dict("records"),
        "daily_nav": daily_nav.replace({np.nan: None}).assign(date=lambda x: x["date"].astype(str)).to_dict("records"),
        "performance_summary": performance_summary.reset_index().replace({np.nan: None}).to_dict("records"),
        "correlation_matrix": correlation_matrix.reset_index().rename(columns={"index": "series"}).replace({np.nan: None}).to_dict("records"),
        "spy_drawdown_events": spy_drawdown_events.replace({np.nan: None}).to_dict("records"),
        "inner_validation_summary": inner_validation_summary.replace({np.nan: None}).to_dict("records"),
        "optimizer_settings": optimizer_settings.replace({np.nan: None}).to_dict("records"),
        "audit_checks": audit_checks.replace({np.nan: None}).to_dict("records"),
        "source_quality": source_quality.replace({np.nan: None}).to_dict("records"),
    },
}
(RUN_DIR / "stage2_payload.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

builder = STAGE2_ROOT / ".artifact_runtime" / "build_stage2_workbook.mjs"
node_candidates = [
    Path(r"C:\Users\Kevin\.cache\codex-runtimes\codex-primary-runtime\dependencies\node\bin\node.exe"),
    Path(shutil.which("node")) if shutil.which("node") else None,
]
node_executable = next((path for path in node_candidates if path and path.is_file()), None)
if builder.is_file() and node_executable:
    completed = subprocess.run(
        [str(node_executable), str(builder), str(RUN_DIR)],
        cwd=builder.parent,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    workbook_path = RUN_DIR / "Stage2_ETF_Portfolio_Results.xlsx"
    preview_path = RUN_DIR / "Stage2_workbook_summary_preview.png"
    if workbook_path.is_file() and preview_path.is_file():
        print("Workbook and preview created successfully.")
    elif completed.returncode != 0:
        print("Workbook builder warning:", completed.stderr)
else:
    print("Workbook builder not available; CSV, JSON, and PNG outputs were still created.")

manifest = {
    "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "methodology": {
        "outer_rule": "One independent target weight vector per holding month using information available by the preceding month-end",
        "inner_rule": "24 one-month validation folds, each trained on the preceding 36 calendar months of weekly total returns",
        "profile_rule": "Choose the risk profile with the highest fixed validation utility over the 24 stitched validation returns",
        "final_rule": "Refit the chosen profile on the latest 36 months, round to 1%, then hold for one month",
        "stage1_winner_matrix": str(WINNER_MATRIX_PATH),
        "stage1_winner_matrix_sha256": file_sha256(WINNER_MATRIX_PATH),
    },
    "audit": audit_checks.to_dict("records"),
    "output_files": [],
}
manifest_path = RUN_DIR / "stage2_run_manifest.json"
for path in sorted(RUN_DIR.iterdir()):
    if path.is_file() and path.name != manifest_path.name:
        manifest["output_files"].append(
            {"name": path.name, "bytes": path.stat().st_size, "sha256": file_sha256(path)}
        )
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = Path(shutil.make_archive(str(OUTPUT_PARENT / RUN_DIR.name), "zip", RUN_DIR))

print("\nStage 2 completed.")
print("Monthly starting weights:", RUN_DIR / "monthly_starting_weights.csv")
print("Performance workbook:", RUN_DIR / "Stage2_ETF_Portfolio_Results.xlsx")
print("Full audit archive:", zip_path)
print("Run folder:", RUN_DIR)


## Main deliverables

- `monthly_starting_weights.csv`: the eight ETFs and beginning-of-month weights for all 121 holding months.
- `daily_portfolio_nav.csv` and `monthly_portfolio_performance.csv`: the stitched out-of-sample path.
- `performance_summary.csv`, correlations, SPY drawdown-event analysis, validation diagnostics, audit checks, and source hashes.
- `Stage2_ETF_Portfolio_Results.xlsx`: a formatted review workbook.
- `stage2_run_manifest.json` and a ZIP package for reproducibility.
